# organizar-trilha-audio.ipynb — Organiza as 3 planilhas de trilha sonora

1. **Freesound_Audio_Stock** -- normaliza `Freesound_Audio_Manager` (filtra só clima de verdade)
2. **Pixabay_YT_Audio_Stock** -- varre suas pastas `audio/<clima>/` no Drive (Pixabay + YouTube Audio Library, os dois juntos -- funcionam do mesmo jeito), parseia o nome de cada arquivo, gera o link tocável
3. **Biblioteca_Match_Audio** -- combina as duas acima na aba `trilha_stock`, que é a que o painel de revisão realmente lê

**Primeira vez rodando**: as 3 planilhas ainda não existem -- esse notebook CRIA elas sozinho e IMPRIME o ID de cada uma. Copie esses IDs pra configuração (célula abaixo) e reuse nas próximas vezes -- senão cria uma planilha nova toda vez!

Não duplica rodando de novo -- atualiza quem já existe, só adiciona o que é novo.

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SETUP                                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U gspread google-api-python-client

from google.colab import drive, auth
from google.auth import default
import gspread
from googleapiclient.discovery import build

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

import shutil
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
_pasta_modulos = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos"
for _arquivo in Path(_pasta_modulos).glob("*.py"):
    shutil.copy(_arquivo, ".")

print("✅ Setup pronto")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup pronto


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# Deixe em branco na 1ª vez -- o notebook cria a planilha e imprime o ID
# aqui embaixo; copie pra cá pra reusar nas próximas vezes.
ID_FREESOUND_AUDIO_STOCK = "1uJj5-Qxs6okWQBi82Xu5Ybjdy8z9kcyFHvmzVukRaBs"
ID_PIXABAY_YT_AUDIO_STOCK = "15knVuLxpZaSXb3ariKPojnNL14eS7AuOuLvKXPh8uEU"
ID_BIBLIOTECA_MATCH_AUDIO = "1VkYaApN1F7X4-52CD0_I-TpKwc2v3XuhUJrIRnZ0Q94"

# Sua planilha de busca automática do Freesound (já existe)
ID_PLANILHA_FREESOUND_MANAGER = "1ieROA_Yy_1fM_qZ_uweLycEJ81l6LwZVJzbYj-sAUA4"  # Freesound_Audio_Manager
NOME_ABA_FREESOUND_MANAGER = "Página1"

# Caminho da pasta no SEU Drive (alanabdmorais@gmail.com) com as subpastas
# por clima -- ex: "audio" (a raiz que contém audio/alegre/, audio/dramatico/...)
CAMINHO_PASTA_AUDIO_DRIVE = "Dark NPOS AlanaBdMorais/Audio"

# URL do Web App publicado a partir do Code.gs (função doGet) -- serve os
# arquivos do Drive através do domínio do Apps Script, contornando o
# bloqueio do Google (desde jan/2024) pra embutir Drive em <audio> de fora
# dele. Publique o script como Web App primeiro (Implantar > Nova implantação
# > Tipo: App da Web > Executar como: Eu > Quem tem acesso: Qualquer pessoa)
# e cole a URL que aparecer aqui.
URL_PROXY_APPS_SCRIPT = "https://script.google.com/macros/s/AKfycbyXbAv5jvc_binNYCij4VdGZl_qHW8SpEfsWvA750s9ENOih4WbgFaYr1thptjXN24G/exec"

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Freesound_Audio_Manager (origem): {ID_PLANILHA_FREESOUND_MANAGER or '(preencher)'}")
print(f"   Pasta de áudio no Drive:          {CAMINHO_PASTA_AUDIO_DRIVE}")
print("=" * 60)

⚙️  CONFIGURAÇÃO
   Freesound_Audio_Manager (origem): 1ieROA_Yy_1fM_qZ_uweLycEJ81l6LwZVJzbYj-sAUA4
   Pasta de áudio no Drive:          Dark NPOS AlanaBdMorais/Audio


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  ABRIR/CRIAR AS 3 PLANILHAS NOVAS                            ║
# ╚══════════════════════════════════════════════════════════════════╝

def abrir_ou_criar_planilha(gc, id_planilha, nome_se_criar):
    """Abre pelo ID se foi passado -- senão CRIA uma nova com esse nome
    e avisa o ID pra você guardar (padrão já usado em abrir_ou_criar_biblioteca_match)."""
    if id_planilha:
        return gc.open_by_key(id_planilha), id_planilha
    nova = gc.create(nome_se_criar)
    print(f"🆕 Planilha '{nome_se_criar}' criada -- ID: {nova.id}")
    print(f"   👉 Copie esse ID pra ID_{nome_se_criar.upper()} na célula de configuração, pra reusar da próxima vez!")
    return nova, nova.id

spreadsheet_freesound_stock, ID_FREESOUND_AUDIO_STOCK = abrir_ou_criar_planilha(
    gc, ID_FREESOUND_AUDIO_STOCK, "Freesound_Audio_Stock")
spreadsheet_pixabay_yt_stock, ID_PIXABAY_YT_AUDIO_STOCK = abrir_ou_criar_planilha(
    gc, ID_PIXABAY_YT_AUDIO_STOCK, "Pixabay_YT_Audio_Stock")
spreadsheet_biblioteca_audio, ID_BIBLIOTECA_MATCH_AUDIO = abrir_ou_criar_planilha(
    gc, ID_BIBLIOTECA_MATCH_AUDIO, "Biblioteca_Match_Audio")

if not ID_PLANILHA_FREESOUND_MANAGER:
    raise ValueError("Preencha ID_PLANILHA_FREESOUND_MANAGER (o ID da sua Freesound_Audio_Manager que já existe).")

print("✅ As 3 planilhas estão abertas/criadas")

✅ As 3 planilhas estão abertas/criadas


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3️⃣  FREESOUND — normaliza a Freesound_Audio_Manager               ║
# ╚══════════════════════════════════════════════════════════════════╝
from trilha_pipeline import (
    carregar_trilha_stock_freesound, carregar_trilha_stock_pasta_drive,
    garantir_aba_trilha_stock, sincronizar_trilha_stock, carregar_trilha_stock_da_planilha,
    achar_pasta_por_caminho,
)

_aba_freesound_manager = gc.open_by_key(ID_PLANILHA_FREESOUND_MANAGER).worksheet(NOME_ABA_FREESOUND_MANAGER)
linhas_freesound = _aba_freesound_manager.get_all_records()
stock_freesound = carregar_trilha_stock_freesound(linhas_freesound)
print(f"📋 Freesound: {len(stock_freesound)} com clima identificado (de {len(linhas_freesound)} linha(s))")

aba_freesound_stock = garantir_aba_trilha_stock(spreadsheet_freesound_stock)
n_novas, n_atualizadas = sincronizar_trilha_stock(aba_freesound_stock, stock_freesound)
print(f"✅ Freesound_Audio_Stock: {n_novas} nova(s), {n_atualizadas} atualizada(s)")

📋 Freesound: 10 com clima identificado (de 20 linha(s))
✅ Freesound_Audio_Stock: 0 nova(s), 10 atualizada(s)


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4️⃣  PIXABAY + YOUTUBE AUDIO LIBRARY — varre as pastas do Drive   ║
# ╚══════════════════════════════════════════════════════════════════╝
# Compartilha cada arquivo como "qualquer pessoa com o link" na primeira
# vez que ele é sincronizado (senão o link tocável dá erro de permissão).

_pasta_raiz_id = achar_pasta_por_caminho(drive_service, CAMINHO_PASTA_AUDIO_DRIVE)
print(f"📁 Pasta '{CAMINHO_PASTA_AUDIO_DRIVE}' encontrada (id: {_pasta_raiz_id})")

stock_pixabay_yt = carregar_trilha_stock_pasta_drive(drive_service, _pasta_raiz_id, tornar_publico=True, url_proxy_apps_script=URL_PROXY_APPS_SCRIPT)
print(f"\n📋 {len(stock_pixabay_yt)} arquivo(s) de áudio encontrados no total")

aba_pixabay_yt_stock = garantir_aba_trilha_stock(spreadsheet_pixabay_yt_stock)
n_novas, n_atualizadas = sincronizar_trilha_stock(aba_pixabay_yt_stock, stock_pixabay_yt)
print(f"✅ Pixabay_YT_Audio_Stock: {n_novas} nova(s), {n_atualizadas} atualizada(s)")

📁 Pasta 'Dark NPOS AlanaBdMorais/Audio' encontrada (id: 1m3olyMdaeg-ubicC077SoYZrh8-8fI70)
   📁 9 subpasta(s) de clima encontrada(s): Desafios x Oportunidades, Desafios, Esperança x Vitória, Clássico, Dramático, Inspirador, Alegre, Melancólico, Calmo 
   📂 Desafios x Oportunidades: 1 arquivo(s)
   📂 Desafios: 1 arquivo(s)
   📂 Esperança x Vitória: 1 arquivo(s)
   📂 Clássico: 0 arquivo(s)
   📂 Dramático: 2 arquivo(s)
   📂 Inspirador: 4 arquivo(s)
   📂 Alegre: 6 arquivo(s)
   📂 Melancólico: 3 arquivo(s)
   📂 Calmo : 2 arquivo(s)

📋 20 arquivo(s) de áudio encontrados no total
✅ Pixabay_YT_Audio_Stock: 0 nova(s), 20 atualizada(s)


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5️⃣  COMBINAR — Biblioteca_Match_Audio.trilha_stock (é essa que  ║
# ║       o painel de revisão lê)                                     ║
# ╚══════════════════════════════════════════════════════════════════╝
pool_freesound = carregar_trilha_stock_da_planilha(aba_freesound_stock)
pool_pixabay_yt = carregar_trilha_stock_da_planilha(aba_pixabay_yt_stock)
pool_combinado = pool_freesound + pool_pixabay_yt

# carregar_trilha_stock_da_planilha() devolve no formato "pronto pro match"
# (tags_clima já expandidas) -- sincronizar_trilha_stock espera o formato
# "cru" (com tags_clima base, ela quem expande) -- aqui as duas coincidem
# o suficiente pra reusar direto (a expansão de algo já expandido não muda nada)

aba_biblioteca_audio = garantir_aba_trilha_stock(spreadsheet_biblioteca_audio)
n_novas, n_atualizadas = sincronizar_trilha_stock(aba_biblioteca_audio, pool_combinado)
print(f"✅ Biblioteca_Match_Audio.trilha_stock: {n_novas} nova(s), {n_atualizadas} atualizada(s)")
print(f"\n📊 Total combinado: {len(pool_combinado)} trilha(s) disponível(is) pro painel de revisão")

✅ Biblioteca_Match_Audio.trilha_stock: 0 nova(s), 30 atualizada(s)

📊 Total combinado: 30 trilha(s) disponível(is) pro painel de revisão
